#### 연습 문제

1. data 폴더 안 rating_train.csv 파일 로드
2. 결측치 제외
3. id 컬럼 제외
4. 중복 데이터 제거
5. train 데이터를 생성하기 위하여 label이 0인 데이터 중 2,000개 추출
6. label이 1인 데이터 중 2,000개 추출
7. 4번, 5번의 결과를 단순 행결합
8. train, test데이터셋을 8:2의 비율로 나눠준다
9. tokenizer는 Okt 사용
10. 불필요한 품사 제외 (사용할 품사: 명사, 동사, 형용사, 부사, 파티클)
11. 글자 수의 제한은 2자리부터 가능
12. tfidf를 사용하여 벡터화
    - `min_df` = 2
    - `ngram_range` = (1,2)
13. 로지스틱 회귀 모델을 사용하여 벡터화한 데이터에서 학습 (random_state = 42)
14. test 데이터셋을 이용하여 검증 후 평가지표 생성
15. 예측 결과, 원본의 데이터셋과 예측 확률을 하나의 데이터프레임으로 생성
16. 결과물 제출: 평가 지표, 15번의 결과에서 상위 5개

In [229]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

from konlpy.tag import Okt

**1**

In [317]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [318]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


**2**

In [319]:
df['document'] = df['document'].str.strip()
df.loc[df['document'] == '', 'document'] = np.nan

In [320]:
df.dropna(inplace=True)

**3**

In [321]:
df.drop('id', axis=1, inplace=True)

**4**

In [322]:
df.drop_duplicates('document', inplace=True)

**5**

In [323]:
df

,document,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,너무재밓었다그래서보는것을추천한다,0
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...
149995,인간이 문제지.. 소는 뭔죄인가..,0
149996,평점이 너무 낮아서...,1
149997,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [324]:
# df_0 = df[df['label'] == 0][:2000].reset_index(drop=True)
df_0 = df.loc[df['label'] == 0, ].iloc[:2000]

In [325]:
df_0

,document,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
2,너무재밓었다그래서보는것을추천한다,0
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
5,막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.,0
6,원작의 긴장감을 제대로 살려내지못했다.,0
...,...,...
4004,노잼 ㅡㅡ보지마세요!,0
4006,진짜 실망하고 본 영화... 매우 안타까움,0
4007,90년대 20년전쯤나왔다면 그나마 흥행했으려나....참 영화만든다는게 나름 열심히한...,0
4010,생 날로 먹을려고 하네 ... 허접영화,0


**6**

In [326]:
# df_1 = df[df['label'] == 1][:2000].reset_index(drop=True)
df_1 = df.loc[df['label'] == 1, ].iloc[:2000]

In [327]:
df_1

,document,label
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
8,액션이 없는데도 재미 있는 몇안되는 영화,1
9,왜케 평점이 낮은건데? 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나?,1
10,걍인피니트가짱이다.진짜짱이다♥,1
...,...,...
4013,20년 전에 이런 영화가 있었다. 즐겁고 재밌고 감동도...,1
4015,되게 웃기다가 되게 울리는 영화,1
4016,잼난다..,1
4017,그냥 한번 보세요 ㅇ ㅇ ? ?,1


**7**

In [328]:
df_concat = pd.concat([df_0, df_1]).reset_index(drop=True)

**8**

In [329]:
df_concat.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   document  4000 non-null   str  
 1   label     4000 non-null   int64
dtypes: int64(1), str(1)
memory usage: 62.6 KB


In [330]:
X = df_concat['document']
y = df_concat['label']

In [244]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**9** ~ **11**

In [245]:
# 9)
okt = Okt()

# 10)
allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb', 'KoreanParticle']

# 11)
len_word = 2

# 토큰화 함수 정의
def tokenize(input_text):
    result = []

    for word, pos in okt.pos(input_text, norm = True, stem = True):
        if (pos in allow_pos) & (len(word) >= len_word):
            result.append(word)
    
    return result

**12**

In [301]:
tfidf = TfidfVectorizer(
    tokenizer = tokenize,
    min_df = 2,
    ngram_range = (1, 2)
)

**13**

In [302]:
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [248]:
print(len(tfidf.get_feature_names_out()))

3866


In [249]:
model = LogisticRegression(random_state = 42)

In [307]:
model.fit(X_train_tf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

**14**

In [251]:
pred = model.predict(X_test_tf)

In [252]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.78      0.84      0.81       400
           1       0.83      0.76      0.79       400

    accuracy                           0.80       800
   macro avg       0.80      0.80      0.80       800
weighted avg       0.80      0.80      0.80       800



**15**

In [253]:
pred

array([1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1,
       0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,

In [254]:
proba = pd.DataFrame(model.predict_proba(X_test_tf), columns=['proba_0', 'proba_1'])
proba

,proba_0,proba_1
0,0.112072,0.887928
1,0.211362,0.788638
2,0.939202,0.060798
3,0.383810,0.616190
4,0.305517,0.694483
...,...,...
795,0.098529,0.901471
796,0.548280,0.451720
797,0.633534,0.366466
798,0.535659,0.464341


In [255]:
pred_df = pd.DataFrame(pred, columns=['pred'])

In [256]:
y_true = pd.DataFrame(y_test.values, columns=['y_true'])

In [257]:
pd.concat([pred_df, y_true, proba], axis=1).sort_values('proba_1', ascending=False).reset_index(drop=True).head()

,pred,y_true,proba_0,proba_1
0,1,1,0.011336,0.988664
1,1,0,0.011336,0.988664
2,1,1,0.011336,0.988664
3,1,1,0.011336,0.988664
4,1,1,0.024585,0.975415


In [258]:
import os
from dotenv import load_dotenv
import requests
import re

In [259]:
load_dotenv()

True

In [260]:
naver_id = os.getenv('naver_api_id')
naver_secret = os.getenv('naver_api_secret')

In [261]:
# 네이버 api를 활용해서 news 제목 수집

url = 'https://openapi.naver.com/v1/search/news.json'
params = {
    'query': '왕사남',
    'display': 30
}

headers = {
    'X-Naver-Client-Id': naver_id,
    'X-Naver-Client-Secret': naver_secret
}

res = requests.get(
    url,
    params = params,
    headers = headers
)

res

<Response [200]>

In [266]:
dict = res.json()

1. res.json()에서 title 부분의 value를 추출하여 하나의 리스트로 생성
2. `<b>`, `</b>` 문자를 제거
3. 위에서 만들어둔 벡터화를 이용하여 벡터화 작업
4. 로지스틱 모델을 이용하여 예측
5. 예측값과 확률을 데이터프레임으로 생성

In [278]:
dict_items = dict['items']

**1**

In [284]:
title_list = []

for i in range(len(dict_items)):
    title_list.append(dict_items[i]['title'])

title_list

['‘<b>왕사남</b>’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다',
 "칸이 열고 관객이 채웠다... '군체', 357만 돌파→'<b>왕사남</b>' 이어 흥행 성...",
 '‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다',
 "'<b>왕사남</b>'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑",
 '단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까',
 '밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행',
 "'<b>왕사남</b>' 촬영지 문경새재, 올들어 153만명 찾았다",
 "'<b>왕사남</b>' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...",
 '반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱',
 "'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",
 "'<b>왕사남</b>' 42년전 '원조 한명회'…배우 故정진, 오늘(2일) 10주기",
 "밀양시, '반하다밀양 반값여행' 얼음골 SNS 이벤트 진행",
 "'<b>왕사남</b>'·'살목지' 다음은 K좀비물…연상호 '군체', 올해 최단 기간 300...",
 '쇼박스는 흥행박스…‘만약에 우리’ ‘<b>왕사남</b>’ ‘살목지’ ‘군체’...',
 "'<b>왕사남</b>' 유영하? &quot;박근혜, 단종처럼 멍에 벗고 복위될 것&quot;",
 "'군체' 주말 극장가 휩쓸고 1위...'<b>왕사남</b>' 이어 천만 넘나?",
 '한강이 연 ‘소설의 시대’ 여전 …상반기 베스트셀러 1~3위 휩쓸어',
 "연상호 감독 '군체', 400만 향해 돌진 중-'<b>왕사남</b>' 넘을까?",
 '유영하 “박근혜, 단종처럼 제자리로 복위될 것”…정치판 ‘<b>왕사남</b>’...',
 "'군체', 주말에만 97만명 봤다…흥행 장기 집권 기대",
 '‘군체’ 인증샷 올린 최휘영 장관 “심장이 쫄깃”',
 "'군체'의 시대...감독·배우 드림팀+ K좀비의 또 다른 시작[MD이슈]",
 '[더벨][매니저 프

**2**

In [289]:
i = 0

for data in title_list:
    title_list[i] = title_list[i].replace('<b>', '')
    title_list[i] = title_list[i].replace('</b>', '')
    i += 1

title_list

['‘왕사남’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다',
 "칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",
 '‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다',
 "'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑",
 '단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까',
 '밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행',
 "'왕사남' 촬영지 문경새재, 올들어 153만명 찾았다",
 "'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...",
 '반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱',
 "'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",
 "'왕사남' 42년전 '원조 한명회'…배우 故정진, 오늘(2일) 10주기",
 "밀양시, '반하다밀양 반값여행' 얼음골 SNS 이벤트 진행",
 "'왕사남'·'살목지' 다음은 K좀비물…연상호 '군체', 올해 최단 기간 300...",
 '쇼박스는 흥행박스…‘만약에 우리’ ‘왕사남’ ‘살목지’ ‘군체’...',
 "'왕사남' 유영하? &quot;박근혜, 단종처럼 멍에 벗고 복위될 것&quot;",
 "'군체' 주말 극장가 휩쓸고 1위...'왕사남' 이어 천만 넘나?",
 '한강이 연 ‘소설의 시대’ 여전 …상반기 베스트셀러 1~3위 휩쓸어',
 "연상호 감독 '군체', 400만 향해 돌진 중-'왕사남' 넘을까?",
 '유영하 “박근혜, 단종처럼 제자리로 복위될 것”…정치판 ‘왕사남’...',
 "'군체', 주말에만 97만명 봤다…흥행 장기 집권 기대",
 '‘군체’ 인증샷 올린 최휘영 장관 “심장이 쫄깃”',
 "'군체'의 시대...감독·배우 드림팀+ K좀비의 또 다른 시작[MD이슈]",
 '[더벨][매니저 프로파일 | 쏠레어파트너스] 영화 현장 20년, 시나리오서...',
 '쌍용C&amp;E·동국대 일산한방병원·강원일보, ‘왕사남의 고장’ 영월에서...'

**3**

In [290]:
title_list_tfidf = tfidf.fit_transform(title_list)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [291]:
model = LogisticRegression(random_state = 42)

In [298]:
# 난 여기가 한계여... 강사님 풀이

new_titles = []

for item in res.json()['items']:
    new_titles.append(item['title'].replace('<b>', '').replace('</b>', ''))

In [303]:
# 벡터화

X_api = tfidf.transform(new_titles)

In [304]:
X_api.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(30, 3866))

In [308]:
pred_api = model.predict(X_api)
proba_api = model.predict_proba(X_api)

In [ ]:
data = []

for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round(max(proba) * 100, 2)
    data.append(
        {
            'title': title,
            'pred': value,
            'pred_proba': proba
        }
    )

df_api = pd.DataFrame(data)

In [310]:
df_api

,title,pred,pred_proba
0,‘왕사남’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다,긍정,54.32
1,"칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",긍정,53.80
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,긍정,72.74
3,'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑,긍정,58.82
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,긍정,67.18
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",긍정,59.13
6,"'왕사남' 촬영지 문경새재, 올들어 153만명 찾았다",긍정,56.32
7,'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...,긍정,54.50
8,"반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱",긍정,52.93
9,"'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",긍정,56.12


In [311]:
df_api.sort_values('pred_proba', ascending=False).head(10)

,title,pred,pred_proba
17,"연상호 감독 '군체', 400만 향해 돌진 중-'왕사남' 넘을까?",긍정,84.26
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,긍정,72.74
26,"‘군체’ 벌써 300만 돌파, ‘왕사남’보다 빠르다 [지금뉴스]",긍정,70.01
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,긍정,67.18
29,"‘군체’, ‘왕사남’보다 빠르다…10일 만에 300만 손익분기점 돌파",긍정,66.15
24,"연간 2위 '군체', 350만명 최단기 기록…'왕사남'보다 2일 빨라",긍정,66.15
21,'군체'의 시대...감독·배우 드림팀+ K좀비의 또 다른 시작[MD이슈],긍정,63.07
19,"'군체', 주말에만 97만명 봤다…흥행 장기 집권 기대",긍정,62.98
16,한강이 연 ‘소설의 시대’ 여전 …상반기 베스트셀러 1~3위 휩쓸어,긍정,60.04
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",긍정,59.13


In [312]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [313]:
ma_scaler = MaxAbsScaler()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline(
    [
        (
            'vec', tfidf
        ),
        (
            'scaler', ma_scaler
        ),
        (
            'model', model
        )
    ]
)

In [314]:
params = {
    'vec__min_df': [2, 3],
    'vec__ngram_range': [(1, 2), (1, 1)],
    'vec__max_features': [None, 1000],
    'model__C': [0.8, 0.9, 1.0]
}

In [316]:
grid = GridSearchCV(
    estimator = pipe,
    param_grid = params,
    cv = cv,
    verbose = 1
)

In [331]:
grid.fit(X, y)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: Use

KeyboardInterrupt: 

In [ ]:
grid.best_params_

In [ ]:
pred_api = grid.predict(new_titles)
proba_api = grid.predict_proba(new_titles)

In [ ]:
data = []

for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round(max(proba) * 100, 2)
    data.append(
        {
            'title': title,
            'pred': value,
            'pred_proba': proba
        }
    )

df_api = pd.DataFrame(data)

In [341]:
load_dotenv()

True

In [342]:
youtube_api = os.getenv('youtube_api')

- 유튜브에서 특정 영상의 댓글을 로드
    - 구글 클라우드 콘솔에서 api 신청
    - 신청된 api key와 영상 id 값 필요
    - 라이브러리 설치
        - google-api-pyhon-client

In [ ]:
# !pip install google-api-python-client

In [337]:
# 영상의 id를 하나 복사 붙여넣기
video_id = 'q6PrfG-yQTo'

In [343]:
from googleapiclient.discovery import build

In [344]:
youtube = build('youtube', 'v3', developerKey = youtube_api)

In [346]:
request = youtube.commentThreads().list(
    part = 'snippet',
    videoId = video_id,
    maxResults = 10,
    textFormat = 'plainText'
)

In [347]:
res = request.execute()

In [348]:
res

{'kind': 'youtube#commentThreadListResponse',
 'etag': 'OXXNBLm46d4h5d8B3FuRjmBQQDc',
 'nextPageToken': 'Z2V0X25ld2VzdF9maXJzdC0tQ2dnSWdBUVZGN2ZST0JJRkNJZ2dHQUFTQlFpSklCZ0FFZ1VJaHlBWUFCSUZDSjBnR0FFU0JRaW9JQmdBSWc0S0RBaVN3NmZJQmhDb3NMN01BUQ==',
 'pageInfo': {'totalResults': 10, 'resultsPerPage': 10},
 'items': [{'kind': 'youtube#commentThread',
   'etag': 'ootnPJdJz3ltBmbmqVEzoiIUG1Y',
   'id': 'Ugwftx9klQb189XAdft4AaABAg',
   'snippet': {'channelId': 'UChY4PXxJI0aJYsGT1X6INWA',
    'videoId': 'q6PrfG-yQTo',
    'topLevelComment': {'kind': 'youtube#comment',
     'etag': '2kvM-mxCsZkUoTlYHkczEjC5a38',
     'id': 'Ugwftx9klQb189XAdft4AaABAg',
     'snippet': {'channelId': 'UChY4PXxJI0aJYsGT1X6INWA',
      'videoId': 'q6PrfG-yQTo',
      'textDisplay': '학교가 명수는 12살 때랑 비슷하넹 세트인가?',
      'textOriginal': '학교가 명수는 12살 때랑 비슷하넹 세트인가?',
      'authorDisplayName': '@슈크림도어가열립-k1z',
      'authorProfileImageUrl': 'https://yt3.ggpht.com/bnsUXep06zD7lxzedZRwdml9OKVy3SKp5Ri7raIITL31fuMwrS-O7IjN2jeiEe

In [356]:
from pprint import pprint

In [361]:
comment = []

for item in res['items']:
    comment.append(item['snippet']['topLevelComment']['snippet']['textDisplay'])

In [362]:
comment

['학교가 명수는 12살 때랑 비슷하넹 세트인가?',
 '여기서 키작은꼬마이야기 가사 받아쓰기20점이 나왔구나ㅋㅋ',
 '진짜 ㅋㅋㅋㅋ 고딩때까지만해도 박명수 맨날 화만 내는 아저씨 이미지로 느껴졌는데 나이 먹고 보니까 제일 웃기네 ㅋㅋㅋㅋㅋㅋㅋ',
 '바보 거지 분장 박명수는 그냥 날라다닌다 ㅋㅋㅋㅋㅋㅋ',
 '요즘 애들이 국어능력이 떨어졌다고하지만 사실은 저때나 지금이나 별 차이가 없다는게 팩트임 ㅋ',
 '허벌가는 지금봐도 웃기넼ㅋㅋㅋㅋㅋㅋㅋㅋㅋ 와 진짜 천재다 ㅋㅋ',
 '세종대왕님 보시면 ㅋㅋㅋ단체능지처참이에요ㅋㅋㅋ',
 '내가 보는 예능 ㅡ 티비로는 자연인, 놀토,  유투브로는 예전 무도',
 '18:17 부터 개웃김',
 '귀여워 ㅋㅋㅋ']

In [363]:
labels = [0, 1, 1, 1, 0, 1, 0, 1, 1, 1]
comment_df = pd.DataFrame(zip(comment, labels), columns = ['review', 'label'])
comment_df

,review,label
0,학교가 명수는 12살 때랑 비슷하넹 세트인가?,0
1,여기서 키작은꼬마이야기 가사 받아쓰기20점이 나왔구나ㅋㅋ,1
2,진짜 ㅋㅋㅋㅋ 고딩때까지만해도 박명수 맨날 화만 내는 아저씨 이미지로 느껴졌는데 나...,1
3,바보 거지 분장 박명수는 그냥 날라다닌다 ㅋㅋㅋㅋㅋㅋ,1
4,요즘 애들이 국어능력이 떨어졌다고하지만 사실은 저때나 지금이나 별 차이가 없다는게 ...,0
5,허벌가는 지금봐도 웃기넼ㅋㅋㅋㅋㅋㅋㅋㅋㅋ 와 진짜 천재다 ㅋㅋ,1
6,세종대왕님 보시면 ㅋㅋㅋ단체능지처참이에요ㅋㅋㅋ,0
7,"내가 보는 예능 ㅡ 티비로는 자연인, 놀토, 유투브로는 예전 무도",1
8,18:17 부터 개웃김,1
9,귀여워 ㅋㅋㅋ,1


In [364]:
tfidf = TfidfVectorizer(
    tokenizer = tokenize,
    min_df = 2,
    ngram_range = (1, 1)
)

ma_scaler = MaxAbsScaler()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(random_state=42, C=0.9)

pipe = Pipeline(
    [
        (
            'vec', tfidf
        ),
        (
            'scaler', ma_scaler
        ),
        (
            'model', model
        )
    ]
)

In [365]:
pipe.fit(X, y)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vec', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function tok...00115DB5FAB90>


In [366]:
pred = pipe.predict(comment)

In [367]:
print(classification_report(pred, labels))

              precision    recall  f1-score   support

           0       0.67      0.40      0.50         5
           1       0.57      0.80      0.67         5

    accuracy                           0.60        10
   macro avg       0.62      0.60      0.58        10
weighted avg       0.62      0.60      0.58        10



In [369]:
pred_proba = model.predict_proba(pred)

ValueError: Expected 2D array, got 1D array instead:
array=[1 0 1 0 0 1 0 0 1 1].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
data = []

for review, label, value, proba in zip(comment, labels, pred, pred_proba):
    label = '긍정' if 